In [78]:
import torch
words = open('../names.txt', 'r').read().splitlines()

In [79]:
N = torch.zeros((27, 27), dtype = torch.int32)
chars = sorted(list(set(''.join(words)))) # 把 words 这个字符串列表里的所有单词，拼接成一个大字符串，中间不加任何东西。
stoi = {s:i+1 for i,s in enumerate(chars)}  # stoi = string to integer
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [80]:
for w in words:
    chs = ['.'] + list(w) + ['.'] 
    for ch1, ch2 in zip(chs, chs[1:]): 
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1  # point

在 note_2 中，我们每一次预测下一个字母都需要进行一次归一化，这一次我们进行优化，把归一化提前，只计算一次。

首先我们让 P 作为一个 27*27 的矩阵 N，
通过 P.sum(1) 沿着第一维度（也就是每一行）求和，
但是我们会发现，维度只有 1 了，但是只有27维才能使得：27*27 的矩阵 / 27的矩阵 = 每一行的归一值，
所以用到keepdim（保持维度），
但即使是 torch.Size([27, 27]) torch.Size([27, 1])，也还是无法计算（需要求出一行27个数的权重，需要有27个被除数），
所以这里就提到 PyTorch广播了，即：
[
    [1],
    [2],
    [3],
]
变成->
[
    [1,1,1,……],
    [2,2,2,……],
    [3,3,3,……],
]

In [ ]:
P = (N+1).float()  # 通过给 N 加一个数，让其可能中不会存在不可能事件，这将使得模型更加的平缓

> 如果你对与这个 +1 只有一个直观的感知，知道所加的数字越大，概率曲线就越平缓（如果把所有概率拿出来画折线图），但是还不满足于当前理解，请看下面。

这个操作有一个优雅的名字：*add-one smoothing / Laplace smoothing（拉普拉斯平滑）*

准确的说，这是给每一种可能的 双字母组合（在本案例中），事先都加一个虚拟计数，让高频事件的概率被压低，低频事件的概率被抬高，从而使得概率分布被拉向均匀分布。

> 其实这里也透露出一个统计学意义：数据越少，越不应该过度相信观察到的频率。

同时，*拉普拉斯平滑*对低计数区域影响大，对高计数区域影响小。

> 其实还表达了一种先验信念：在没有足够证据之前，不认为任何一种可能性绝对不可能。

所以从比较准确的统计意义上来理解，应该是：加一平滑把经验分布向均匀分布做收**缩（shrinkage）**，降低极端概率，尤其抑制小样本下的过度自信。

In [94]:
print(P)
print(P.sum())
print(P.sum(1))
print(P.sum(1, keepdim = True))
print(P.shape, P.sum(1, keepdim = True).shape)

tensor([[1.0000e+00, 4.4110e+03, 1.3070e+03, 1.5430e+03, 1.6910e+03, 1.5320e+03,
         4.1800e+02, 6.7000e+02, 8.7500e+02, 5.9200e+02, 2.4230e+03, 2.9640e+03,
         1.5730e+03, 2.5390e+03, 1.1470e+03, 3.9500e+02, 5.1600e+02, 9.3000e+01,
         1.6400e+03, 2.0560e+03, 1.3090e+03, 7.9000e+01, 3.7700e+02, 3.0800e+02,
         1.3500e+02, 5.3600e+02, 9.3000e+02],
        [6.6410e+03, 5.5700e+02, 5.4200e+02, 4.7100e+02, 1.0430e+03, 6.9300e+02,
         1.3500e+02, 1.6900e+02, 2.3330e+03, 1.6510e+03, 1.7600e+02, 5.6900e+02,
         2.5290e+03, 1.6350e+03, 5.4390e+03, 6.4000e+01, 8.3000e+01, 6.1000e+01,
         3.2650e+03, 1.1190e+03, 6.8800e+02, 3.8200e+02, 8.3500e+02, 1.6200e+02,
         1.8300e+02, 2.0510e+03, 4.3600e+02],
        [1.1500e+02, 3.2200e+02, 3.9000e+01, 2.0000e+00, 6.6000e+01, 6.5600e+02,
         1.0000e+00, 1.0000e+00, 4.2000e+01, 2.1800e+02, 2.0000e+00, 1.0000e+00,
         1.0400e+02, 1.0000e+00, 5.0000e+00, 1.0600e+02, 1.0000e+00, 1.0000e+00,
         8.4300e+

In [95]:
# P = P / P.sum(1, keepdim=True) # 创建新 Tensor，再让 P 指向它。所以会额外申请一块结果内存
P /= P.sum(1, keepdim=True)  # 直接在原来的 P 上原地修改
P # 求出了归一化的 tensor对象

# 当然，你也可以尝试 去掉 keepdim=True （因为默认是False），
# 从后面的 P.sum(1) 观察，并思考相信你会对 行归一化（一行中，和为1） 和 列归一化（一列中，和为1） 有一些认知，
# P.sum(1)也是一种检验方法，因为，如果我们忘记 保持维度 ，代码不会报错，而我们计算出来的概率的确是错的

tensor([[3.1192e-05, 1.3759e-01, 4.0767e-02, 4.8129e-02, 5.2745e-02, 4.7785e-02,
         1.3038e-02, 2.0898e-02, 2.7293e-02, 1.8465e-02, 7.5577e-02, 9.2452e-02,
         4.9064e-02, 7.9195e-02, 3.5777e-02, 1.2321e-02, 1.6095e-02, 2.9008e-03,
         5.1154e-02, 6.4130e-02, 4.0830e-02, 2.4641e-03, 1.1759e-02, 9.6070e-03,
         4.2109e-03, 1.6719e-02, 2.9008e-02],
        [1.9583e-01, 1.6425e-02, 1.5983e-02, 1.3889e-02, 3.0756e-02, 2.0435e-02,
         3.9809e-03, 4.9835e-03, 6.8796e-02, 4.8685e-02, 5.1899e-03, 1.6779e-02,
         7.4575e-02, 4.8213e-02, 1.6039e-01, 1.8872e-03, 2.4475e-03, 1.7988e-03,
         9.6279e-02, 3.2997e-02, 2.0288e-02, 1.1264e-02, 2.4623e-02, 4.7771e-03,
         5.3963e-03, 6.0480e-02, 1.2857e-02],
        [4.3039e-02, 1.2051e-01, 1.4596e-02, 7.4850e-04, 2.4701e-02, 2.4551e-01,
         3.7425e-04, 3.7425e-04, 1.5719e-02, 8.1587e-02, 7.4850e-04, 3.7425e-04,
         3.8922e-02, 3.7425e-04, 1.8713e-03, 3.9671e-02, 3.7425e-04, 3.7425e-04,
         3.1549e-

In [96]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    ix = [0]
    out = []
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples = 1, replacement = True, generator = g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.


In [97]:
print(P[0, :].sum()) # 第0行的所有列
print(P[:, 0].sum()) # 所有行的第0列

tensor(1.)
tensor(3.0023)


后面我们尝试思考模型质量以及如何评估他

目标：
最大化数据相对于模型参数的似然性  
等价于最大化对数似然  
等价于最小化负对数似然  
等价于最小化平均负对数似然

In [98]:
log_likelihood = 0.0  # 对数似然估计
n = 0
# for w in words:
for w in ["justki"]:
    chs = ['.'] + list(w) + ['.'] 
    for ch1, ch2 in zip(chs, chs[1:]): 
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)   # log(x) from 0 to 1
        log_likelihood += logprob
        n += 1
        print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')
print(f'{log_likelihood = }')

nll = -log_likelihood  # 负对数似然（因为我们希望得到一个损失函数，而损失函数有一个特点：少即是好）
print(f'{nll = }') 
print(f'{nll/n = }')  # 平均对数似然（这就是通常意义上的损失函数），模型训练的工作是找到 最小化负对数似然损失 的 参数

.j: 0.0756 -2.5826
ju: 0.0694 -2.6685
us: 0.1502 -1.8956
st: 0.0942 -2.3625
tk: 0.0002 -8.6300
ki: 0.1007 -2.2961
i.: 0.1405 -1.9629
log_likelihood = tensor(-22.3982)
nll = tensor(22.3982)
nll/n = tensor(3.1997)
